
# Ejercicios — Métricas Avanzadas

## Universidad de Navarra · Programa PwC · 2026

**Tiempo recomendado: 2 horas**

Este notebook está pensado para trabajar durante la parte práctica de la sesión. No buscamos implementar métodos complejos: el objetivo es **calcular, visualizar e interpretar métricas**.

### Entregable

Para cada ejercicio, entrega tu notebook con el código ejecutado y una breve interpretación de los resultados.


In [ ]:

# ==============================================
# Configuración común
# ==============================================
import warnings
warnings.filterwarnings('ignore')

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from urllib.request import urlopen, Request
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, r2_score,
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, classification_report
)

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True


def cargar_desde_url(url, tipo='csv'):
    """Carga un CSV o Excel desde una URL."""
    if tipo == 'csv':
        return pd.read_csv(url)
    if tipo == 'excel':
        return pd.read_excel(url)
    raise ValueError("tipo debe ser 'csv' o 'excel'")


def columnas_numericas_utiles(df):
    """Selecciona columnas numéricas y excluye identificadores evidentes por nombre."""
    num = df.select_dtypes(include=np.number).copy()
    id_tokens = ('id', 'index', 'indice', 'codigo', 'code', 'serial')
    drop = []
    for c in num.columns:
        name = str(c).strip().lower().replace('-', '_').replace(' ', '_')
        if name in {'id', 'index', 'indice'} or any(tok in name for tok in id_tokens):
            drop.append(c)
    usable = [c for c in num.columns if c not in drop]
    return num[usable] if usable else num



# Ejercicio 1 — PCA + K-Means: ¿se puede dividir el mundo en dos grupos?

### Dataset

**UN Development Indicators**

```text
https://github.com/manoelgadi/PSDMA/raw/refs/heads/main/datasets/UN.csv
```

### Preguntas de investigación

1. **¿Puede el mundo dividirse de forma significativa en solo dos clusters (desarrollados vs. en desarrollo)?**
2. **¿Cuántos clusters sugieren los datos según la Elbow Rule?**
3. **¿Cuánta estructura de los datos explican dos clusters frente al número óptimo de clusters?**
4. **¿La evidencia apoya la afirmación de _Factfulness_ de que el mundo no puede dividirse en solo dos grupos, sino en 4?**



## Parte 1 — Preparación de datos

Tu notebook debe incluir:

### 1. Exploración
- Carga del dataset.
- Inspección de variables.
- Identificación de indicadores numéricos.

### 2. Valores perdidos
- Sustituye los valores perdidos por la **mediana**.

### 3. Escalado
- Utiliza `StandardScaler`.
- Explica brevemente por qué el escalado es necesario antes de clustering.

### 4. Número de clusters
- Aplica K-Means.
- Prueba varios valores de `k`.
- Construye la **Elbow Curve**.
- Propón un `k` recomendado.



## Parte 2 — Comparar k = 2 con el número óptimo

Construye dos soluciones:

- **k = 2**
- **k = k óptimo** según tu análisis de la Elbow Curve

Compara las dos soluciones.

### Mínimo que debes mostrar

- El valor elegido de `k`.
- La Elbow Curve.
- Número de observaciones en cada cluster.
- Una visualización 2D utilizando las dos primeras componentes de PCA.
- Una breve interpretación relacionada con las preguntas de investigación.


In [ ]:

UN_URL = 'https://github.com/manoelgadi/PSDMA/raw/refs/heads/main/datasets/UN.csv'
un = cargar_desde_url(UN_URL, 'csv')
un.head()


In [ ]:

# TODO: completa el código
numeric_un = un.select_dtypes(include=np.number).copy()
numeric_un = numeric_un.fillna(numeric_un.median(numeric_only=True))

scaler = StandardScaler()
X_un = scaler.fit_transform(numeric_un)

# Prueba varios k y calcula la inercia
ks = range(2, 9)
inertias = []
for k in ks:
    # TODO: crea y ajusta KMeans
    pass

# TODO: crea la Elbow Curve


### Interpretación
Escribe 5–8 líneas respondiendo a las cuatro preguntas de investigación.


# Ejercicio 2 — Linear Regression con HBAT

### Dataset

```text
https://github.com/manoelgadi/PSDMA/raw/refs/heads/main/datasets/HBAT.xlsx
```

Utiliza una variable objetivo **continua** de HBAT. Una opción natural es `satisfac` (satisfacción del cliente), pero puedes justificar otra variable continua.

### Objetivo
Construir una regresión lineal sencilla y centrar el análisis en:

- **SSE**
- **MSE**
- **R²**

### Tareas

1. Carga el Excel.
2. Identifica las variables numéricas disponibles.
3. Selecciona una variable objetivo continua.
4. Divide en train/test.
5. Entrena `LinearRegression` de `scikit-learn`.
6. Calcula SSE, MSE y R² sobre el conjunto de test.
7. Genera una gráfica de **real vs. predicho**.
8. Explica en lenguaje de negocio qué significa tu R².


In [ ]:

HBAT_URL = 'https://github.com/manoelgadi/PSDMA/raw/refs/heads/main/datasets/HBAT.xlsx'
hbat = cargar_desde_url(HBAT_URL, 'excel')
hbat.head()


In [ ]:

# TODO: selecciona la variable objetivo continua y los predictores.
# Pista: puedes empezar por las variables numéricas de percepción de HBAT.

numeric_hbat = hbat.select_dtypes(include=np.number).copy()
numeric_hbat = numeric_hbat.fillna(numeric_hbat.median(numeric_only=True))

X = numeric_hbat[['x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13','x14', 'x15', 'x16', 'x17']]
y = numeric_hbat['x18']

# TODO: ajustar LinearRegression y calcula SSE, MSE y R²


### Interpretación
Explica qué información aporta cada una de las tres métricas y cuál utilizarías para comunicar el desempeño del modelo a un directivo.


# Ejercicio 3 — Logistic Regression con IRIS

### Dataset

```text
load iris
```

Para este ejercicio utilizaremos `alianza` como variable objetivo binaria: **consideraría / no consideraría una alianza estratégica con HBAT**.

### Objetivo
Construir una clasificación binaria y analizar:

- Matriz de confusión
- Accuracy
- Precision
- Recall
- F1-score
- AUC
- GINI
- KS
- McFadden pseudo-R² y McFadden ajustado como medida complementaria



## Tareas

1. Carga IRIS.
2. Entrena `LogisticRegression`.
3. Obtén probabilidades con `predict_proba()`.
4. Utiliza un umbral inicial de **0.50** para generar clases.
5. Calcula la matriz de confusión.
6. Genera un `classification_report`.
7. Calcula Accuracy, Precision, Recall y F1.
8. Construye ROC y calcula AUC.
8. Calcula GINI con 
$$2\cdot AUC-1$$.
10. Calcula KS como 
$$\max|TPR-FPR|$$.
11. Explica qué métrica utilizarías como principal si el coste de un **falso negativo** fuera muy elevado.


In [ ]:
import seaborn as sns
import pandas as pd
from sklearn.linear_model import LogisticRegression



In [ ]:
from sklearn.datasets import load_iris
data = load_iris()


In [ ]:
df = pd.DataFrame(data['data'], columns=data['feature_names'])


In [ ]:
df['target'] = data['target']


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop('target', axis=1), df['target'], test_size=0.3)


In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)


In [ ]:

# TODO: Matriz de confusión y classification_report
# TODO: Accuracy, Precision, Recall y F1
# TODO: ROC + AUC
# TODO: GINI
# TODO: KS


### Interpretación
Escribe una conclusión de 8–10 líneas. Explica especialmente la diferencia entre **probabilidad**, **umbral** y **clasificación final**.


# Ejercicio 4 — Métrica correcta para la pregunta correcta

Para cada caso, indica qué métrica priorizarías y explica por qué.

### Caso A
Una entidad quiere reducir el número de clientes solventes rechazados por error.

### Caso B
Un modelo de riesgo quiere separar correctamente buenos y malos clientes en todo el rango de puntos de corte.

### Caso C
Una regresión predice el precio de una vivienda y la dirección quiere saber qué proporción de la variabilidad está recogiendo el modelo.

### Caso D
Quieres reducir el número de variables de un dataset manteniendo la mayor parte de la información posible.

### Caso E
Necesitas agrupar clientes y quieres justificar visualmente por qué eliges 4 clusters en lugar de 8.

> No existe una métrica “ganadora” para todos los problemas. **La métrica debe seguir a la pregunta de negocio.**



# ✅ Checklist de entrega

### Ejercicio 1 — PCA + K-Means
- [ ] Dataset cargado
- [ ] Missing values tratados
- [ ] StandardScaler
- [ ] Elbow Curve
- [ ] k = 2
- [ ] k óptimo
- [ ] Comparación e interpretación

### Ejercicio 2 — Linear Regression
- [ ] Train/test
- [ ] SSE
- [ ] MSE
- [ ] R²
- [ ] Gráfica real vs predicho
- [ ] Interpretación

### Ejercicio 3 — Logistic Regression
- [ ] Matriz de confusión
- [ ] Classification report
- [ ] Accuracy
- [ ] Precision
- [ ] Recall
- [ ] F1
- [ ] ROC / AUC
- [ ] GINI
- [ ] KS
- [ ] Pseudo-R²
- [ ] Interpretación

### Ejercicio 4 — Métrica vs. pregunta
- [ ] Justificación de cada elección
